# Gold archetype cluster assignments

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import s3fs
from collections import defaultdict
import re
from datetime import date, timedelta

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.ml.archetype_clustering import (
    numeric_feature_columns,
    prepare_dataframe_for_archetype_clustering,
)

pd.set_option("display.max_columns", None)

## Batter

In [ ]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"  # default GOLD_PREFIX in src/pipeline/settings.py

role = "batter"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_archetypes.parquet"
with fs.open(path, "rb") as f:
    df_batter_archetypes = pd.read_parquet(f)

pd.set_option("display.max_columns", None)
df_batter_archetypes.head()

### Batter cluster profiles

Mean values of `numeric_feature_columns()` (same exclusions as PCA / GMM) by `cluster_id`. Use this table to pick short labels for each cluster. Edit `batter_cluster_names` below to record them.

In [ ]:
_df_b = prepare_dataframe_for_archetype_clustering(df_batter_archetypes)
_feat_b = [c for c in numeric_feature_columns(_df_b) if c != "cluster_id"]
_feat_b = [c for c in _feat_b if c in _df_b.columns]

_gb_b = _df_b.groupby("cluster_id", sort=True)
batter_n_players = _gb_b.size().rename("n_players")
batter_feature_means_by_cluster = _gb_b[_feat_b].mean()

batter_cluster_names: dict[int, str] = {
    # e.g. 0: "high-whiff chase profile",
}

print("Players per cluster")
display(batter_n_players.to_frame())
print("Mean features by cluster (rows = features, columns = cluster_id)")
display(batter_feature_means_by_cluster.T)
if batter_cluster_names:
    print("Same table with your labels as column names")
    display(
        batter_feature_means_by_cluster.rename(columns=batter_cluster_names).T
    )

## Pitcher

In [ ]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"

role = "pitcher"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_archetypes.parquet"
with fs.open(path, "rb") as f:
    df_pitcher_archetypes = pd.read_parquet(f)

df_pitcher_archetypes.head()

### Pitcher cluster profiles

Mean values of `numeric_feature_columns()` by `cluster_id` for pitchers. Edit `pitcher_cluster_names` to record labels.

In [ ]:
_df_p = prepare_dataframe_for_archetype_clustering(df_pitcher_archetypes)
_feat_p = [c for c in numeric_feature_columns(_df_p) if c != "cluster_id"]
_feat_p = [c for c in _feat_p if c in _df_p.columns]

_gb_p = _df_p.groupby("cluster_id", sort=True)
pitcher_n_players = _gb_p.size().rename("n_players")
pitcher_feature_means_by_cluster = _gb_p[_feat_p].mean()

pitcher_cluster_names: dict[int, str] = {
    # e.g. 0: "low-velo command",
}

print("Players per cluster")
display(pitcher_n_players.to_frame())
print("Mean features by cluster (rows = features, columns = cluster_id)")
display(pitcher_feature_means_by_cluster.T)
if pitcher_cluster_names:
    print("Same table with your labels as column names")
    display(
        pitcher_feature_means_by_cluster.rename(columns=pitcher_cluster_names).T
    )